# Binary OLE `.tbx` Reverse Engineering — Corpus Analysis

**Goal:** Build a corpus of binary ArcGIS Desktop `.tbx` files alongside their XML ArcGIS Pro
counterparts to systematically decode the OLE binary format without an ArcGIS Desktop install.

**Strategy:**
- Each `_arcmap.tbx` in `solutions-geoprocessing-toolbox` has a matching `_pro.tbx` (XML) for the
  **same toolbox** — the XML is the semantic *answer key* for the binary.
- We supplement with the local file `GAFY09 Toolbox v2.tbx` from this workspace.
- Phases: container recon → entropy/string extraction → XML ground-truth correlation → struct hypotheses → schema documentation.

**Corpus sources:**
| # | Binary (_arcmap) | XML Reference (_pro) | Tool Count | Notes |
|---|---|---|---|---|
| 1 | `Incident Analysis Tools_arcmap.tbx` | `Incident Analysis Tools_pro.tbx` | ~4 | Complex DAG, preconditions |
| 2 | `Geonames Tools_arcmap.tbx` | `Geonames Tools_pro.tbx` | ~2 | Script tools |
| 3 | `Distance To Assets_arcmap.tbx` | `Distance To Assets_pro.tbx` | ~3 | Hardcoded path examples |
| 4 | `GAFY09 Toolbox v2.tbx` | *(none — local only)* | ? | Target file for migration |

## Phase 0 — Setup: Install Dependencies & Configure Paths

In [1]:
import subprocess, sys

# Ensure required packages are available in this kernel
for pkg in ["olefile", "requests"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import olefile
import requests
import os
import re
import math
import struct
import collections
import xml.etree.ElementTree as ET
from pathlib import Path

print(f"olefile {olefile.__version__}")
print(f"Python  {sys.version.split()[0]}")

# ── Paths ────────────────────────────────────────────────────────────────────
WORKSPACE   = Path(r"D:\01_dev\a-Arc2Qgis")
CORPUS_DIR  = WORKSPACE / "research" / "corpus"
LOCAL_TBX   = WORKSPACE / "arcgis-tools" / "GAFY09 Toolbox v2.tbx"

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nCorpus dir : {CORPUS_DIR}")
print(f"Local .tbx : {LOCAL_TBX}  exists={LOCAL_TBX.exists()}")

olefile 0.47
Python  3.11.9

Corpus dir : D:\01_dev\a-Arc2Qgis\research\corpus
Local .tbx : D:\01_dev\a-Arc2Qgis\arcgis-tools\GAFY09 Toolbox v2.tbx  exists=True


## Phase 1 — Download Corpus

Download paired `_arcmap.tbx` (binary) and `_pro.tbx` (XML) from
`Esri/solutions-geoprocessing-toolbox` on GitHub. The raw download URL pattern is:

```
https://raw.githubusercontent.com/Esri/solutions-geoprocessing-toolbox/dev/<folder>/<filename>
```

Each pair gives us the **binary** to decode and the **XML answer key** to verify against.

In [2]:
BASE = "https://raw.githubusercontent.com/Esri/solutions-geoprocessing-toolbox/dev"

# (folder, binary_filename, xml_filename, short_label)
CORPUS_MANIFEST = [
    ("incident_analysis",
     "Incident Analysis Tools_arcmap.tbx",
     "Incident Analysis Tools_pro.tbx",
     "incident_analysis"),
    ("geonames",
     "Geonames Tools_arcmap.tbx",
     "Geonames Tools_pro.tbx",
     "geonames"),
    ("distance_to_assets",
     "Distance To Assets_arcmap.tbx",
     "Distance To Assets_pro.tbx",
     "distance_to_assets"),
    ("sun_position_analysis",
     "Sun Position Analysis Tools_arcmap.tbx",
     "Sun Position Analysis Tools_pro.tbx",
     "sun_position"),
    ("military_features",
     "Military Features Tools_arcmap.tbx",
     None,                                    # Pro version may not exist
     "military_features"),
]

def _download(url: str, dest: Path, label: str) -> bool:
    """Download url → dest, skip if already present. Returns True on success."""
    if dest.exists():
        print(f"  [skip]  {label} ({dest.stat().st_size:,} bytes already on disk)")
        return True
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        dest.write_bytes(r.content)
        print(f"  [ok]    {label}  →  {dest.stat().st_size:,} bytes")
        return True
    except Exception as exc:
        print(f"  [FAIL]  {label}  →  {exc}")
        return False


print("Downloading corpus ...\n")
corpus = []   # list of dicts: {label, binary, xml}

for folder, bin_name, xml_name, label in CORPUS_MANIFEST:
    entry = {"label": label, "binary": None, "xml": None}

    bin_url  = f"{BASE}/{requests.utils.quote(folder)}/{requests.utils.quote(bin_name)}"
    bin_dest = CORPUS_DIR / f"{label}_arcmap.tbx"
    if _download(bin_url, bin_dest, f"{label} binary"):
        entry["binary"] = bin_dest

    if xml_name:
        xml_url  = f"{BASE}/{requests.utils.quote(folder)}/{requests.utils.quote(xml_name)}"
        xml_dest = CORPUS_DIR / f"{label}_pro.tbx"
        if _download(xml_url, xml_dest, f"{label} XML"):
            entry["xml"] = xml_dest

    corpus.append(entry)

# Add local file (binary only — no XML pair)
corpus.append({"label": "GAFY09_local", "binary": LOCAL_TBX, "xml": None})

print(f"\nCorpus ready: {sum(1 for e in corpus if e['binary'])} binary, "
      f"{sum(1 for e in corpus if e['xml'])} XML pairs")


  [ok]    incident_analysis binary  →  1,614,336 bytes
  [ok]    incident_analysis XML  →  5,548,544 bytes
  [ok]    geonames binary  →  286,208 bytes
  [ok]    geonames XML  →  291,840 bytes
  [ok]    distance_to_assets binary  →  2,463,232 bytes
  [ok]    distance_to_assets XML  →  3,450,880 bytes
  [ok]    sun_position binary  →  67,072 bytes
  [ok]    sun_position XML  →  65,024 bytes
  [ok]    military_features binary  →  233,984 bytes

Corpus ready: 6 binary, 4 XML pairs


## Phase 2 — OLE Container Reconnaissance

For each binary file:
1. Verify the `D0 CF 11 E0` magic bytes.
2. Enumerate all storages (folders) and streams (data blobs) with sizes.
3. Read OLE metadata (`SummaryInformation` stream) — author, app, timestamps.

This maps the internal directory tree before any payload bytes are examined.

In [3]:
OLE_MAGIC = b"\xd0\xcf\x11\xe0"

def check_magic(path: Path) -> bool:
    with open(path, "rb") as f:
        return f.read(4) == OLE_MAGIC


def ole_recon(path: Path) -> dict:
    """Full OLE directory enumeration for a single .tbx file."""
    result = {"path": path, "streams": [], "meta": {}}
    with olefile.OleFileIO(str(path)) as ole:
        # --- Directory listing ---
        for entry in ole.listdir(streams=True, storages=True):
            stream_path = "/".join(entry)
            try:
                data = ole.openstream(entry).read()
                size = len(data)
            except Exception:
                size = -1
            result["streams"].append({"path": stream_path, "size": size})

        # --- OLE metadata ---
        try:
            meta = ole.get_metadata()
            result["meta"] = {
                "author":          getattr(meta, "author",           b"").decode("latin-1", errors="replace"),
                "last_saved_by":   getattr(meta, "last_saved_by",    b"").decode("latin-1", errors="replace"),
                "creating_app":    getattr(meta, "creating_application", b"").decode("latin-1", errors="replace"),
                "create_time":     str(getattr(meta, "create_time",  "")),
                "last_saved_time": str(getattr(meta, "last_saved_time", "")),
            }
        except Exception as e:
            result["meta"]["error"] = str(e)
    return result


# Run reconnaissance on every binary in the corpus
recon_results = {}

print(f"{'File':<30}  {'Magic':6}  Streams")
print("-" * 65)

for entry in corpus:
    if entry["binary"] is None or not entry["binary"].exists():
        continue
    path  = entry["binary"]
    label = entry["label"]
    ok    = check_magic(path)
    if ok:
        info = ole_recon(path)
        recon_results[label] = info
        n = len(info["streams"])
        print(f"{label:<30}  {'OLE':6}  {n} streams")
    else:
        # Might be XML
        with open(path, "rb") as f:
            header = f.read(8)
        print(f"{label:<30}  XML?   {header[:4]}")

print(f"\n{len(recon_results)} binary OLE files processed.")

File                            Magic   Streams
-----------------------------------------------------------------
incident_analysis               OLE     8 streams
geonames                        OLE     6 streams
distance_to_assets              OLE     7 streams
sun_position                    OLE     3 streams
military_features               OLE     11 streams
GAFY09_local                    OLE     3 streams

6 binary OLE files processed.


In [4]:
import pprint

# Pretty-print one recon result as a reference
if recon_results:
    first_label = next(iter(recon_results))
    info = recon_results[first_label]
    print(f"=== {first_label} ===")
    print("\n-- Metadata --")
    pprint.pprint(info["meta"])
    print("\n-- Stream Directory --")
    print(f"  {'Stream Path':<50}  {'Size':>8}")
    print("  " + "-" * 62)
    for s in sorted(info["streams"], key=lambda x: x["path"]):
        print(f"  {s['path']:<50}  {s['size']:>8,}")

=== incident_analysis ===

-- Metadata --
{'error': "'NoneType' object has no attribute 'decode'"}

-- Stream Directory --
  Stream Path                                             Size
  --------------------------------------------------------------
  Contents                                               5,891
  Tool0                                                350,070
  Tool1                                                113,807
  Tool2                                                448,036
  Tool3                                                116,588
  Tool4                                                381,206
  Tool5                                                180,049
  Version                                                   66


## Phase 3 — Entropy & String Extraction

For each stream in each binary:
- Compute **Shannon entropy** to classify content type.
- Extract **ASCII** and **UTF-16-LE** strings (min 5 chars).
- Flag streams containing known ESRI semantic anchors (`GPFeatureLayer`, `GPString`, tool names).

Low entropy + readable strings = highest value targets.

In [5]:
ESRI_ANCHORS = [
    b"GPFeatureLayer", b"GPString", b"GPDouble", b"GPLong", b"GPBoolean",
    b"GPRasterLayer", b"GPTable", b"GPMultiValue", b"GPValueTable",
    b"ModelBuilder", b"esriGeoprocessing", b"GPToolbox",
    b"GetParameterAsText", b"SetParameterAsText",
    b"Input", b"Output", b"Required", b"Optional", b"Derived",
]

def shannon_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = collections.Counter(data)
    total  = len(data)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def extract_strings(data: bytes, min_len: int = 5):
    """Return (ascii_hits, utf16le_hits) as decoded string lists."""
    ascii_hits  = [m.decode("ascii",    errors="replace")
                   for m in re.findall(rb"[\x20-\x7e]{%d,}" % min_len, data)]
    # UTF-16LE: alternating printable + null byte
    utf16_hits  = [m.decode("utf-16-le", errors="replace")
                   for m in re.findall(rb"(?:[\x20-\x7e]\x00){%d,}" % min_len, data)]
    return ascii_hits, utf16_hits


def anchor_hits(data: bytes) -> list[str]:
    return [a.decode() for a in ESRI_ANCHORS if a.lower() in data.lower()]


def analyse_streams(label: str, path: Path) -> list[dict]:
    """Return per-stream analysis rows for one .tbx file."""
    rows = []
    with olefile.OleFileIO(str(path)) as ole:
        for entry in ole.listdir(streams=True):
            try:
                data = ole.openstream(entry).read()
            except Exception:
                continue
            stream_name = "/".join(entry)
            entropy     = shannon_entropy(data)
            ascii_s, u16_s = extract_strings(data)
            anchors     = anchor_hits(data)
            rows.append({
                "stream":   stream_name,
                "size":     len(data),
                "entropy":  round(entropy, 2),
                "ascii_n":  len(ascii_s),
                "utf16_n":  len(u16_s),
                "anchors":  anchors,
                "ascii_sample":  ascii_s[:5],
                "utf16_sample":  u16_s[:5],
            })
    return rows


# Build full analysis table
print(f"{'Label':<25} {'Stream':<50} {'Size':>7} {'H':>5} {'A':>4} {'U':>4} {'Anchors'}")
print("-" * 115)

stream_analysis = {}
for entry in corpus:
    if entry["binary"] is None or not entry["binary"].exists():
        continue
    if not check_magic(entry["binary"]):
        continue
    label = entry["label"]
    rows  = analyse_streams(label, entry["binary"])
    stream_analysis[label] = rows
    for r in rows:
        anchor_str = ", ".join(r["anchors"][:3]) + ("…" if len(r["anchors"]) > 3 else "")
        print(f"{label:<25} {r['stream']:<50} {r['size']:>7,} {r['entropy']:>5.2f} "
              f"{r['ascii_n']:>4} {r['utf16_n']:>4}  {anchor_str}")

Label                     Stream                                                Size     H    A    U Anchors
-------------------------------------------------------------------------------------------------------------------
incident_analysis         Contents                                             5,891  3.98    2   25  
incident_analysis         Tool0                                              350,070  4.56  562 4150  Input, Output, Required
incident_analysis         Tool1                                              113,807  4.82  302 1175  Input, Output, Required…
incident_analysis         Tool2                                              448,036  4.73  982 6285  Input, Output, Required…
incident_analysis         Tool3                                              116,588  4.37  162 1320  Input, Output, Required…
incident_analysis         Tool4                                              381,206  5.36 1836 2881  Input, Output, Required
incident_analysis         Tool5        

In [6]:
# Focus: print the highest-value streams (high anchor count, lower entropy)
print("\n── High-Value Streams (has ESRI anchors) ──\n")
for label, rows in stream_analysis.items():
    for r in rows:
        if r["anchors"]:
            print(f"  [{label}]  {r['stream']}  size={r['size']:,}  H={r['entropy']}  anchors={r['anchors']}")
            if r["utf16_sample"]:
                print(f"    UTF-16 sample: {r['utf16_sample'][:3]}")
            if r["ascii_sample"]:
                print(f"    ASCII  sample: {r['ascii_sample'][:3]}")


── High-Value Streams (has ESRI anchors) ──

  [incident_analysis]  Tool0  size=350,070  H=4.56  anchors=['Input', 'Output', 'Required']
    UTF-16 sample: ['FindPercentChange', 'Find Percent Change', 'This tool calculates the number of incidents in one, two, several, or all categories in a specified time range, and within an area of interest, compared to the number of incident in those same categories in a previous time period.']
    ASCII  sample: ['PROJCS["WGS_1984_Web_Mercator_Auxiliary_Sphere",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Mercator_Auxiliary_Sphere"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",0.0],PARAMETER["Standard_Parallel_1",0.0],PARAMETER["Auxiliary_Sphere_Type",0.0],UNIT["Meter",1.0],AUTHORITY["EPSG",3857]]', 'PROJCS["WGS_1984_Web_Mercator_Auxiliary_Sphere",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SP

## Phase 4 — XML Ground-Truth Correlation

Parse each `_pro.tbx` XML answer key to extract the **complete semantic schema**:
- Tool names and display names
- Parameter names, types, directions, defaults
- Connection graph (tool call order)

Then locate each of these known strings in the corresponding binary streams
to establish **byte offset → semantic meaning** mappings.

In [7]:
def parse_xml_tbx(path: Path) -> dict:
    """
    Parse a Pro .tbx XML file (root: <GPToolbox>) and return a structured
    schema dict: {toolbox_name, tools: [{name, display_name, params: [...]}]}.
    """
    schema = {"toolbox_name": "", "tools": []}
    try:
        tree = ET.parse(str(path))
        root = tree.getroot()
    except ET.ParseError as e:
        schema["parse_error"] = str(e)
        return schema

    # Toolbox name — varies by schema version; try common tag names
    for tag in ("Name", "DisplayName", "AliasName"):
        node = root.find(f".//{tag}")
        if node is not None and node.text:
            schema["toolbox_name"] = node.text.strip()
            break

    # Iterate tools — tag is usually <GPProcess> or <GPTool>
    for tool_node in root.iter("GPProcess"):
        tool = {
            "name":         tool_node.findtext("Name", "").strip(),
            "display_name": tool_node.findtext("DisplayName", "").strip(),
            "params":       [],
        }
        for param in tool_node.iter("GPParameter"):
            tool["params"].append({
                "name":         param.findtext("Name",        "").strip(),
                "display_name": param.findtext("DisplayName", "").strip(),
                "data_type":    param.findtext("DataType/TypeName",  "").strip(),
                "direction":    param.findtext("Direction",   "").strip(),
                "param_type":   param.findtext("ParameterType", "").strip(),
            })
        schema["tools"].append(tool)

    return schema


# Parse all XML pairs and build ground-truth corpus
xml_schemas = {}
for entry in corpus:
    if entry["xml"] and entry["xml"].exists():
        label  = entry["label"]
        schema = parse_xml_tbx(entry["xml"])
        xml_schemas[label] = schema
        n_tools  = len(schema["tools"])
        n_params = sum(len(t["params"]) for t in schema["tools"])
        print(f"[{label}]  toolbox='{schema['toolbox_name']}'  "
              f"tools={n_tools}  params_total={n_params}")
        for t in schema["tools"]:
            print(f"  Tool: {t['name']!r} ({t['display_name']!r})")
            for p in t["params"]:
                print(f"    {p['name']:<30}  {p['data_type']:<20}  {p['direction']:<10}  {p['param_type']}")
        print()

[incident_analysis]  toolbox=''  tools=0  params_total=0

[geonames]  toolbox=''  tools=0  params_total=0

[distance_to_assets]  toolbox=''  tools=0  params_total=0

[sun_position]  toolbox=''  tools=0  params_total=0



In [ ]:
def locate_anchors_in_binary(label: str, bin_path: Path, schema: dict) -> list[dict]:
    """
    For each known string from the XML schema, locate every occurrence in
    every OLE stream of the binary and record the byte offset.
    Returns a list of hit dicts for cross-stream analysis.
    """
    # Build the set of known-meaning strings from the XML schema
    known_strings = set()
    known_strings.add(schema["toolbox_name"])
    for tool in schema["tools"]:
        known_strings.update([tool["name"], tool["display_name"]])
        for p in tool["params"]:
            known_strings.update([p["name"], p["display_name"], p["data_type"]])
    known_strings.discard("")

    hits = []
    with olefile.OleFileIO(str(bin_path)) as ole:
        for entry in ole.listdir(streams=True):
            try:
                data = ole.openstream(entry).read()
            except Exception:
                continue
            stream_name = "/".join(entry)
            for s in known_strings:
                if not s:
                    continue
                # Search as UTF-16-LE (OLE/COM default for wide strings)
                needle_u16 = s.encode("utf-16-le")
                pos = 0
                while True:
                    idx = data.find(needle_u16, pos)
                    if idx == -1:
                        break
                    hits.append({
                        "string": s, "encoding": "utf-16-le",
                        "stream": stream_name, "offset": idx,
                        "context_hex": data[max(0,idx-4):idx+len(needle_u16)+8].hex(" "),
                    })
                    pos = idx + 1

                # Also search as ASCII
                needle_asc = s.encode("ascii", errors="ignore")
                pos = 0
                while True:
                    idx = data.find(needle_asc, pos)
                    if idx == -1:
                        break
                    hits.append({
                        "string": s, "encoding": "ascii",
                        "stream": stream_name, "offset": idx,
                        "context_hex": data[max(0,idx-4):idx+len(needle_asc)+8].hex(" "),
                    })
                    pos = idx + 1
    return hits


# Run correlation for each paired entry
correlation = {}
for entry in corpus:
    label = entry["label"]
    if label not in xml_schemas or entry["binary"] is None:
        continue
    if not entry["binary"].exists() or not check_magic(entry["binary"]):
        continue
    hits = locate_anchors_in_binary(label, entry["binary"], xml_schemas[label])
    correlation[label] = hits
    print(f"[{label}]  {len(hits)} anchor hits across all streams")
    for h in hits[:8]:
        print(f"  stream={h['stream']!r:<40}  offset=0x{h['offset']:06x}  "
              f"enc={h['encoding']:<8}  str={h['string']!r}")

## Phase 5 — Struct Hypothesis & Prefix Analysis

**Key insight:** COM/OLE serializes strings as `BSTR` (length-prefixed wide strings):

```
4-byte LE length (byte count, not char count) | UTF-16-LE chars
```

For each anchor hit (UTF-16-LE), inspect the 4 bytes **before** it.
If `struct.unpack('<I', bytes_before)[0] == len(string) * 2`, the hypothesis is confirmed.

This confirms whether ESRI's serializer uses standard BSTR, a custom length prefix, 
or some other delimiter — and at what exact byte offset each field begins.

In [ ]:
def test_bstr_hypothesis(bin_path: Path, hits: list[dict]) -> list[dict]:
    """
    For each UTF-16-LE anchor hit, read the 4 bytes before it and test
    multiple length-prefix hypotheses:
      - BSTR:    prefix == byte_length (len * 2)
      - WSTR:    prefix == char_length (len)
      - BSTR+2:  prefix == byte_length + 2  (some ESRI variants add 2)
      - None:    no valid prefix pattern detected
    Returns enhanced hit list with 'prefix_type' and 'prefix_value'.
    """
    results = []
    stream_cache = {}

    with olefile.OleFileIO(str(bin_path)) as ole:
        for entry in ole.listdir(streams=True):
            try:
                stream_cache["/".join(entry)] = ole.openstream(entry).read()
            except Exception:
                pass

    for h in hits:
        if h["encoding"] != "utf-16-le":
            continue
        data    = stream_cache.get(h["stream"])
        if data is None:
            continue
        offset  = h["offset"]
        s       = h["string"]
        byte_len = len(s.encode("utf-16-le"))
        char_len = len(s)

        prefix_type = "unknown"
        prefix_val  = None

        if offset >= 4:
            prefix_bytes = data[offset - 4 : offset]
            pv = struct.unpack("<I", prefix_bytes)[0]
            prefix_val = pv
            if pv == byte_len:
                prefix_type = "BSTR (byte_len)"
            elif pv == char_len:
                prefix_type = "WSTR (char_len)"
            elif pv == byte_len + 2:
                prefix_type = "BSTR+2"
            elif pv == byte_len - 2:
                prefix_type = "BSTR-2"

        results.append({**h, "prefix_type": prefix_type, "prefix_val": prefix_val,
                        "byte_len": byte_len, "char_len": char_len})
    return results


# Tally hypothesis confirmation across all files
tally = collections.Counter()
all_prefix_results = {}

for label, hits in correlation.items():
    entry = next(e for e in corpus if e["label"] == label)
    results = test_bstr_hypothesis(entry["binary"], hits)
    all_prefix_results[label] = results
    for r in results:
        tally[r["prefix_type"]] += 1

print("── Prefix Type Distribution (all files, UTF-16-LE hits) ──\n")
for ptype, count in tally.most_common():
    bar = "█" * min(count, 60)
    print(f"  {ptype:<25}  {count:>5}  {bar}")

# Show confirmed BSTR hits as examples
print("\n── Sample confirmed BSTR hits ──\n")
for label, results in all_prefix_results.items():
    confirmed = [r for r in results if "BSTR" in r["prefix_type"]][:4]
    if confirmed:
        print(f"[{label}]")
        for r in confirmed:
            print(f"  stream={r['stream']!r}  offset=0x{r['offset']:06x}  "
                  f"prefix={r['prefix_val']!r}  type={r['prefix_type']!r}  str={r['string']!r}")

## Phase 6 — Cross-File Stream Comparison

Compare the **stream directory structure** across all binary files to identify:
- Streams that appear in **every** file (likely core schema streams)
- Streams that vary by tool count (likely per-tool record blocks)
- Streams unique to specific toolboxes

This identifies which stream(s) to focus parsing effort on.

In [ ]:
# Build a set of stream names per file
stream_sets = {
    label: {r["path"] for r in rows}
    for label, rows in stream_analysis.items()
}

all_stream_names = set()
for s in stream_sets.values():
    all_stream_names |= s

n_files = len(stream_sets)
print(f"Comparing stream directories across {n_files} binary files\n")
print(f"{'Stream Path':<55}  {'Present in':>10}  Ubiquitous?")
print("-" * 80)

stream_presence = {}
for sname in sorted(all_stream_names):
    count = sum(1 for s in stream_sets.values() if sname in s)
    stream_presence[sname] = count
    ubiq = "★ CORE" if count == n_files else ""
    print(f"  {sname:<53}  {count:>3}/{n_files:<3}      {ubiq}")

core_streams = [s for s, c in stream_presence.items() if c == n_files]
print(f"\n★ Core streams (in all {n_files} files): {core_streams}")

## Phase 7 — Deep Hex Dump of Core Streams

For each **core stream** (present in all files), dump the first 256 bytes as a 
formatted hex+ASCII view alongside the known semantic content from the XML.

This is where the structural layout of fields becomes visible.

In [ ]:
def hex_dump(data: bytes, offset: int = 0, width: int = 16, max_rows: int = 24) -> str:
    """Return a formatted hex+ASCII dump string."""
    lines = []
    for i in range(0, min(len(data), max_rows * width), width):
        chunk   = data[i:i + width]
        hex_str = " ".join(f"{b:02x}" for b in chunk).ljust(width * 3)
        asc_str = "".join(chr(b) if 0x20 <= b < 0x7f else "." for b in chunk)
        lines.append(f"  {offset + i:06x}  {hex_str}  {asc_str}")
    return "\n".join(lines)


def read_stream(bin_path: Path, stream_path: str) -> bytes:
    parts = stream_path.split("/")
    with olefile.OleFileIO(str(bin_path)) as ole:
        return ole.openstream(parts).read()


# Dump the first 256 bytes of each core stream from every file
for sname in core_streams[:4]:   # limit to first 4 core streams to keep output manageable
    print(f"\n{'='*70}")
    print(f"  Stream: {sname!r}")
    print(f"{'='*70}")
    for entry in corpus:
        label = entry["label"]
        bpath = entry["binary"]
        if bpath is None or not bpath.exists() or not check_magic(bpath):
            continue
        data = read_stream(bpath, sname)
        print(f"\n  ── [{label}]  total={len(data):,} bytes ──")
        print(hex_dump(data, max_rows=16))

## Phase 8 — Schema Registry: Findings Log

As hypotheses are confirmed through Phases 5–7, record them here.
This cell should be **manually updated** as discoveries are made.

> **Instructions:** After running Phases 5–7, record confirmed field offsets,
> stream names, encoding patterns, and struct layouts below.
> Each entry becomes a building block for the final parser.

In [ ]:
# ── Schema Registry ─────────────────────────────────────────────────────────
# Populate this dict as findings are confirmed in Phases 5–7.
# Structure: {stream_name: {field_name: {offset, length, encoding, confirmed_by}}}

SCHEMA_REGISTRY = {
    # Example (to be replaced with actual findings):
    # "Contents": {
    #     "toolbox_name": {
    #         "offset_hypothesis": "variable — BSTR at start of stream",
    #         "encoding": "utf-16-le",
    #         "prefix": "BSTR (4-byte LE byte count)",
    #         "confirmed_by": ["incident_analysis", "geonames"],
    #     }
    # }
}

print("Schema Registry initialized.")
print("Run Phases 5–7, then populate SCHEMA_REGISTRY with confirmed findings.")
print(f"\nCurrent entries: {len(SCHEMA_REGISTRY)}")